# Course 2: Privacy-Preserving Machine Learning
## Session 2: Differential privacy from first principles

**Goal:** release private counts and means from a synthetic health table and plot the error against epsilon.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. A synthetic health table
Every row is one invented patient.

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(0)
n = 2000
df = pd.DataFrame({"age": rng.integers(18, 90, n), "diabetes": rng.random(n) < 0.12,
                   "systolic_bp": np.clip(rng.normal(128, 18, n), 80, 220)})
print(df.head(3)); print("true diabetes count:", int(df.diabetes.sum()), "| true mean blood pressure: %.2f" % df.systolic_bp.mean())

### 2. The Laplace mechanism
Adding or removing one patient changes a count by at most 1. That number is the sensitivity. Adding Laplace noise with scale `sensitivity / epsilon` gives epsilon-DP.

In [ ]:
def laplace_count(values, eps, rng):
    return values.sum() + rng.laplace(scale=1.0 / eps)

for eps in [0.1, 1.0, 10.0]:
    print(f"epsilon {eps:5.1f}: private count {laplace_count(df.diabetes, eps, rng):8.1f}   (expected absolute error {1 / eps:.1f})")

### 3. A private mean needs bounds
First clip every value to a fixed range, so one person can move the sum by at most the width of the range. Then split the budget between a noisy sum and a noisy count.

In [ ]:
def private_mean(x, lo, hi, eps, rng):
    x = np.clip(x, lo, hi)
    noisy_sum = (x - lo).sum() + rng.laplace(scale=(hi - lo) / (eps / 2))
    noisy_n = len(x) + rng.laplace(scale=1 / (eps / 2))
    return lo + noisy_sum / max(noisy_n, 1.0)

print("true mean %.2f | private mean at epsilon 1: %.2f" % (df.systolic_bp.mean(), private_mean(df.systolic_bp.values, 80, 220, 1.0, rng)))

### 4. Error against epsilon

In [ ]:
import matplotlib.pyplot as plt
eps_grid = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]
err_count = [np.mean([abs(laplace_count(df.diabetes, e, rng) - df.diabetes.sum()) for _ in range(500)]) for e in eps_grid]
err_mean = [np.mean([abs(private_mean(df.systolic_bp.values, 80, 220, e, rng) - df.systolic_bp.mean()) for _ in range(500)]) for e in eps_grid]
for e, a, b in zip(eps_grid, err_count, err_mean): print(f"epsilon {e:5.2f}   count error {a:8.2f}   mean error {b:7.3f}")
assert err_count[0] > err_count[-1] and abs(err_count[4] - 1.0) < 0.2          # at epsilon 1 the expected error is 1
plt.loglog(eps_grid, err_count, "o-", label="count"); plt.loglog(eps_grid, err_mean, "s-", label="mean")
plt.xlabel("epsilon"); plt.ylabel("mean absolute error"); plt.legend(); plt.savefig("error_vs_epsilon.png")

### 5. The Gaussian mechanism and delta
Gaussian noise gives (epsilon, delta)-DP. A simple and valid calibration goes through zero-concentrated DP: noise `sigma` on a query with L2 sensitivity `s` gives `rho = s² / (2 sigma²)`, and that implies `epsilon = rho + 2 sqrt(rho ln(1/delta))`.

In [ ]:
from math import log, sqrt
def gaussian_sigma(sensitivity, eps, delta):
    L = log(1 / delta)
    rho = (sqrt(L + eps) - sqrt(L)) ** 2          # solves eps = rho + 2 sqrt(rho L)
    return sensitivity / sqrt(2 * rho)

for eps in [0.5, 1, 4]: print(f"epsilon {eps}: sigma for a count = {gaussian_sigma(1, eps, 1e-6):.2f}")

### 6. Composition: budgets add up
Answering k questions with epsilon each costs k times epsilon in total under basic composition. With Gaussian noise the rho values add, which is much cheaper for many questions.

In [ ]:
k, eps_each, delta = 100, 0.1, 1e-6
L = log(1 / delta)
rho_each = (sqrt(L + eps_each) - sqrt(L)) ** 2
rho_total = k * rho_each
print("basic composition    : total epsilon =", k * eps_each)
print("zCDP composition     : total epsilon = %.2f (at delta 1e-6)" % (rho_total + 2 * sqrt(rho_total * L)))
assert rho_total + 2 * sqrt(rho_total * L) < k * eps_each

### 7. Check the definition by experiment
Two neighbouring tables differ in one patient. For any output, the two probabilities may differ by a factor of at most e^epsilon.

In [ ]:
eps = 1.0
a = 240 + rng.laplace(scale=1 / eps, size=500_000)       # count on table D
b = 241 + rng.laplace(scale=1 / eps, size=500_000)       # count on D plus one patient
bins = np.arange(236, 247)
pa, pb = np.histogram(a, bins)[0] / len(a), np.histogram(b, bins)[0] / len(b)
worst = np.max(np.maximum(pa / pb, pb / pa))
print("largest probability ratio seen: %.3f   bound e^epsilon = %.3f" % (worst, np.e ** eps))
assert worst < np.e ** eps * 1.05

### TODO
1. Release a private histogram of age in 10-year bins. What is the sensitivity of the whole histogram?
2. Clip blood pressure to [100, 160] instead. The noise drops but a bias appears. Plot both effects.
3. A colleague suggests running the private count 1,000 times and averaging. Explain what that does to the privacy guarantee.